# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 个人GitHub项目说明

- 每名学生独立完成本Notebook；
- 输入文件固定为`data/E Commerce Dataset.xlsx`；
- 输出固定写入`output/day04_project/`；
- 不要提交教师演示Notebook或教师参考答案；
- 完成后重启内核并从头运行，再推送到个人GitHub仓库。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [1]:
# 1. 项目初始化与数据读取

import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)  # 修正拼写错误
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

def find_project_root(start=None):
    """从当前目录向上寻找种子项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    # 修正语法：正确遍历当前目录及其所有父目录
    for candidate in [start] + list(start.parents):
        if (candidate / "data" / "E Commerce Dataset.xlsx").exists():
            return candidate
    raise FileNotFoundError("未找到data/E Commerce Dataset.xlsx")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "E Commerce Dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据: {DATA_PATH}")
print(f"项目输出目录: {OUTPUT_DIR}")
print(f"原始数据形状: {raw_df.shape}")
raw_df.head()

原始数据: D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24040087\data\E Commerce Dataset.xlsx
项目输出目录: D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24040087\output\day04_project
原始数据形状: (5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [2]:
# 在此写下你的答案：
# 1.代表每一个独立用户的购买记录
# 2.Churn
# 3.1. 缺乏业务含义、会导致模型过拟合、违反机器学习假设

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [3]:
def build_quality_report(data):
    """返回字段级数据质量报告（核心版）。

    Args:
        data: DataFrame，输入数据

    Returns:
        DataFrame，包含数据类型、缺失数量、缺失比例、唯一值数量
    """
    report = pd.DataFrame({
        '数据类型': data.dtypes,
        '缺失数量': data.isnull().sum(),
        '缺失比例(%)': (data.isnull().sum() / len(data) * 100).round(2),
        '唯一值数量': data.nunique()
    })

    return report

# 使用核心版本
quality_before = build_quality_report(raw_df)
print("=== 清洗前数据质量报告 ===")
print(quality_before)

# 保存报告
quality_before.to_csv(OUTPUT_DIR / 'quality_report_before.csv')

=== 清洗前数据质量报告 ===
                                数据类型  缺失数量  缺失比例(%)  唯一值数量
CustomerID                     int64     0     0.00   5630
Churn                          int64     0     0.00      2
Tenure                       float64   264     4.69     36
PreferredLoginDevice             str     0     0.00      3
CityTier                       int64     0     0.00      3
WarehouseToHome              float64   251     4.46     34
PreferredPaymentMode             str     0     0.00      7
Gender                           str     0     0.00      2
HourSpendOnApp               float64   255     4.53      6
NumberOfDeviceRegistered       int64     0     0.00      6
PreferedOrderCat                 str     0     0.00      6
SatisfactionScore              int64     0     0.00      5
MaritalStatus                    str     0     0.00      3
NumberOfAddress                int64     0     0.00     15
Complain                       int64     0     0.00      2
OrderAmountHikeFromlastYear  float64  

### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [4]:
# 如果 "PreferredPaymentMode" 或 "PreferredOrderCat" 列名不同
# 先查看实际列名，然后调整

print("=" * 60)
print("项目初始审计报告")
print("=" * 60)

# 1. 完全重复行数
duplicate_rows = raw_df.duplicated().sum()
print(f"\\n1. 完全重复行数: {duplicate_rows}")

# 2. CustomerID 重复数量
customer_id_duplicates = raw_df['CustomerID'].duplicated().sum()
print(f"\\n2. CustomerID 重复数量: {customer_id_duplicates}")

# 3. Churn 的频数和流失率
print(f"\\n3. Churn 频数分布:")
churn_counts = raw_df['Churn'].value_counts()
print(churn_counts)
churn_rate = (raw_df['Churn'].sum() / len(raw_df) * 100).round(2)
print(f"   流失率: {churn_rate}%")

# 4. 主要类别字段的频数
# 先获取所有object类型列
object_cols = raw_df.select_dtypes(include=['object', 'string']).columns.tolist()
print(f"\\n4. 所有类别型字段: {object_cols}")

# 检查特定的类别列
category_cols_to_check = ["PreferredLoginDevice", "PreferredPayment", "PreferredPaymentMode", "PreferredOrderCat"]
for col in category_cols_to_check:
    if col in raw_df.columns:
        print(f"\\n   【{col}】频数分布:")
        print(raw_df[col].value_counts(dropna=False))
    else:
        # 寻找相似的列名
        similar_cols = [c for c in raw_df.columns if col.lower() in c.lower()]
        if similar_cols:
            print(f"\\n   【{col}】未找到，相似的列: {similar_cols}")
        else:
            print(f"\\n   【{col}】未找到")

项目初始审计报告
\n1. 完全重复行数: 0
\n2. CustomerID 重复数量: 0
\n3. Churn 频数分布:
Churn
0    4682
1     948
Name: count, dtype: int64
   流失率: 16.84%
\n4. 所有类别型字段: ['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender', 'PreferedOrderCat', 'MaritalStatus']
\n   【PreferredLoginDevice】频数分布:
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64
\n   【PreferredPayment】未找到，相似的列: ['PreferredPaymentMode']
\n   【PreferredPaymentMode】频数分布:
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64
\n   【PreferredOrderCat】未找到


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [5]:
# 1. 项目初始化与数据读取
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start] + list(start.parents):
        if (candidate / "data" / "E Commerce Dataset.xlsx").exists():
            return candidate
    raise FileNotFoundError("未找到data/E Commerce Dataset.xlsx")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "E Commerce Dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")
print(f"原始数据形状: {raw_df.shape}")


# 2. 数据质量报告
def build_quality_report(data):
    return pd.DataFrame({
        '数据类型': data.dtypes,
        '缺失数量': data.isnull().sum(),
        '缺失比例(%)': (data.isnull().sum() / len(data) * 100).round(2),
        '唯一值数量': data.nunique()
    })

quality_before = build_quality_report(raw_df)
print("\\n清洗前质量报告:")
print(quality_before)
quality_before.to_csv(OUTPUT_DIR / 'quality_report_before.csv')


# 3. 初始审计
print("\\n=== 初始审计 ===")
print(f"完全重复行数: {raw_df.duplicated().sum()}")
print(f"CustomerID重复数量: {raw_df['CustomerID'].duplicated().sum()}")
print("\\nChurn分布:")
print(raw_df['Churn'].value_counts())
print(f"流失率: {(raw_df['Churn'].sum() / len(raw_df) * 100).round(2)}%")

category_cols = ["PreferredLoginDevice", "PreferredPaymentMode", "PreferredOrderCat"]
for col in category_cols:
    if col in raw_df.columns:
        print(f"\\n{col}:")
        print(raw_df[col].value_counts(dropna=False))


# 4. 清洗规则
NUMERIC_MISSING_COLS = ["Tenure", "WarehouseToHome", "HourSpendOnApp",
                        "OrderAmountHikeFromLastYear", "CouponUsed", "OrderCount", "DaySinceLastOrder"]

CATEGORY_MAPPINGS = {}
if "PreferredLoginDevice" in raw_df.columns:
    CATEGORY_MAPPINGS["PreferredLoginDevice"] = {"Phone": "Mobile Phone"}
if "PreferredPaymentMode" in raw_df.columns:
    CATEGORY_MAPPINGS["PreferredPaymentMode"] = {"COD": "Cash on Delivery", "CC": "Credit Card"}
elif "PreferredPayment" in raw_df.columns:
    CATEGORY_MAPPINGS["PreferredPayment"] = {"COD": "Cash on Delivery", "CC": "Credit Card"}
if "PreferredOrderCat" in raw_df.columns:
    CATEGORY_MAPPINGS["PreferredOrderCat"] = {"Mobile": "Mobile Phone", "Phone": "Mobile Phone"}


# 5. 数据清洗（修复ChainedAssignmentError）
def clean_data(df):
    df_clean = df.copy()
    log = []

    # 删除重复行
    dup = df_clean.duplicated().sum()
    if dup > 0:
        df_clean = df_clean.drop_duplicates()
        log.append(f"删除 {dup} 行重复数据")

    # 类别映射 - 使用赋值方式
    for col, mapping in CATEGORY_MAPPINGS.items():
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].replace(mapping)
            log.append(f"{col} 完成类别统一")

    # 中位数填补缺失值 - 使用赋值方式替代inplace
    for col in NUMERIC_MISSING_COLS:
        if col in df_clean.columns:
            missing_count = df_clean[col].isnull().sum()
            if missing_count > 0:
                median = df_clean[col].median()
                df_clean[col] = df_clean[col].fillna(median)
                log.append(f"{col} 用中位数 {median:.2f} 填补了 {missing_count} 个缺失值")

    return df_clean, log

df_cleaned, log = clean_data(raw_df)
print("\\n=== 清洗日志 ===")
for item in log:
    print(f"✓ {item}")


# 6. 清洗后报告
quality_after = build_quality_report(df_cleaned)
print("\\n清洗后质量报告:")
print(quality_after)
quality_after.to_csv(OUTPUT_DIR / 'quality_report_after.csv')

print("\\n清洗前后缺失对比:")
print(pd.DataFrame({'清洗前': quality_before['缺失数量'], '清洗后': quality_after['缺失数量']}))

df_cleaned.to_csv(OUTPUT_DIR / 'e_commerce_cleaned.csv', index=False)
print(f"\\n清洗后数据已保存至: {OUTPUT_DIR / 'e_commerce_cleaned.csv'}")
print(f"清洗后数据形状: {df_cleaned.shape}")

原始数据形状: (5630, 20)
\n清洗前质量报告:
                                数据类型  缺失数量  缺失比例(%)  唯一值数量
CustomerID                     int64     0     0.00   5630
Churn                          int64     0     0.00      2
Tenure                       float64   264     4.69     36
PreferredLoginDevice             str     0     0.00      3
CityTier                       int64     0     0.00      3
WarehouseToHome              float64   251     4.46     34
PreferredPaymentMode             str     0     0.00      7
Gender                           str     0     0.00      2
HourSpendOnApp               float64   255     4.53      6
NumberOfDeviceRegistered       int64     0     0.00      6
PreferedOrderCat                 str     0     0.00      6
SatisfactionScore              int64     0     0.00      5
MaritalStatus                    str     0     0.00      3
NumberOfAddress                int64     0     0.00     15
Complain                       int64     0     0.00      2
OrderAmountHikeFromlastYea

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [6]:
# 4. 编写可复用清洗函数

NUMERIC_MISSING_COLS = ["Tenure", "WarehouseToHome", "HourSpendOnApp",
                        "OrderAmountHikeFromLastYear", "CouponUsed", "OrderCount", "DaySinceLastOrder"]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {"Phone": "Mobile Phone"},
    "PreferredPaymentMode": {"COD": "Cash on Delivery", "CC": "Credit Card"},
    "PreferredOrderCat": {"Mobile": "Mobile Phone", "Phone": "Mobile Phone"}
}


def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    Returns:
        cleaned_df: 清洗后的DataFrame
        cleaning_log: 日志DataFrame
    """
    df = data.copy()
    logs = []
    initial_rows = len(df)

    # 1. 删除完全重复行
    dup = df.duplicated().sum()
    if dup > 0:
        df = df.drop_duplicates()
    logs.append({
        '处理步骤': '删除重复行',
        '处理规则': '删除完全重复记录',
        '处理前记录数': initial_rows,
        '处理后记录数': len(df),
        '影响记录数': dup
    })

    # 2. 类别标准化
    for col, mapping in CATEGORY_MAPPINGS.items():
        if col in df.columns:
            affected = df[col].isin(mapping.keys()).sum()
            if affected > 0:
                df[col] = df[col].replace(mapping)
            logs.append({
                '处理步骤': f'类别标准化-{col}',
                '处理规则': f'统一 {list(mapping.keys())}',
                '处理前记录数': len(df),
                '处理后记录数': len(df),
                '影响记录数': affected
            })

    # 3. 中位数填补缺失值
    for col in NUMERIC_MISSING_COLS:
        if col in df.columns:
            missing = df[col].isnull().sum()
            if missing > 0:
                median = df[col].median()
                df[col] = df[col].fillna(median)
            logs.append({
                '处理步骤': f'缺失值填补-{col}',
                '处理规则': f'中位数填补',
                '处理前记录数': len(df),
                '处理后记录数': len(df),
                '影响记录数': missing
            })

    # 4. 数据类型转换
    for col in ['Churn', 'Complain']:
        if col in df.columns:
            df[col] = df[col].astype('int')

    return df, pd.DataFrame(logs)


# 执行清洗
df_cleaned, cleaning_log = clean_ecommerce_data(raw_df)

print("\\n清洗日志:")
print(cleaning_log.to_string(index=False))

print(f"\\n清洗前: {len(raw_df)} 行, 清洗后: {len(df_cleaned)} 行")

quality_after = build_quality_report(df_cleaned)
print("\\n清洗后质量报告:")
print(quality_after)

df_cleaned.to_csv(OUTPUT_DIR / 'e_commerce_cleaned.csv', index=False)
cleaning_log.to_csv(OUTPUT_DIR / 'cleaning_log.csv', index=False)
print(f"\\n清洗后数据已保存至: {OUTPUT_DIR / 'e_commerce_cleaned.csv'}")

\n清洗日志:
                      处理步骤             处理规则  处理前记录数  处理后记录数  影响记录数
                     删除重复行         删除完全重复记录    5630    5630      0
类别标准化-PreferredLoginDevice     统一 ['Phone']    5630    5630   1231
类别标准化-PreferredPaymentMode 统一 ['COD', 'CC']    5630    5630    638
              缺失值填补-Tenure            中位数填补    5630    5630    264
     缺失值填补-WarehouseToHome            中位数填补    5630    5630    251
      缺失值填补-HourSpendOnApp            中位数填补    5630    5630    255
          缺失值填补-CouponUsed            中位数填补    5630    5630    256
          缺失值填补-OrderCount            中位数填补    5630    5630    258
   缺失值填补-DaySinceLastOrder            中位数填补    5630    5630    307
\n清洗前: 5630 行, 清洗后: 5630 行
\n清洗后质量报告:
                                数据类型  缺失数量  缺失比例(%)  唯一值数量
CustomerID                     int64     0     0.00   5630
Churn                          int64     0     0.00      2
Tenure                       float64     0     0.00     36
PreferredLoginDevice             str     0     0

### 任务 3：运行清洗函数并查看日志

In [7]:
# 任务 3：运行清洗函数并查看日志

# 执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

# 显示清洗日志
print("\\n" + "="*60)
print("清洗日志:")
print("="*60)
display(cleaning_log)

# 显示清洗后数据预览
print("\\n" + "="*60)
print("清洗后数据预览:")
print("="*60)
cleaned_df.head()

# 显示清洗后数据形状
print(f"\\n清洗后数据形状: {cleaned_df.shape}")

# 验证清洗效果
print("\\n" + "="*60)
print("清洗效果验证:")
print("="*60)

# 检查是否还有重复行
print(f"剩余重复行数: {cleaned_df.duplicated().sum()}")

# 检查缺失值是否已处理
print("\\n剩余缺失值:")
print(cleaned_df.isnull().sum()[cleaned_df.isnull().sum() > 0])

# 检查类别是否已标准化
print("\\n类别标准化验证:")
for col in CATEGORY_MAPPINGS.keys():
    if col in cleaned_df.columns:
        print(f"\\n{col} 唯一值:")
        print(cleaned_df[col].unique())

# 检查数据类型
print("\\n数据类型:")
print(cleaned_df.dtypes)

\n============================================================
清洗日志:


,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除重复行,删除完全重复记录,5630,5630,0
1,类别标准化-PreferredLoginDevice,统一 ['Phone'],5630,5630,1231
2,类别标准化-PreferredPaymentMode,"统一 ['COD', 'CC']",5630,5630,638
3,缺失值填补-Tenure,中位数填补,5630,5630,264
4,缺失值填补-WarehouseToHome,中位数填补,5630,5630,251
5,缺失值填补-HourSpendOnApp,中位数填补,5630,5630,255
6,缺失值填补-CouponUsed,中位数填补,5630,5630,256
7,缺失值填补-OrderCount,中位数填补,5630,5630,258
8,缺失值填补-DaySinceLastOrder,中位数填补,5630,5630,307


\n============================================================
清洗后数据预览:
\n清洗后数据形状: (5630, 20)
\n============================================================
清洗效果验证:
剩余重复行数: 0
\n剩余缺失值:
OrderAmountHikeFromlastYear    265
dtype: int64
\n类别标准化验证:
\nPreferredLoginDevice 唯一值:
<StringArray>
['Mobile Phone', 'Computer']
Length: 2, dtype: str
\nPreferredPaymentMode 唯一值:
<StringArray>
['Debit Card', 'UPI', 'Credit Card', 'Cash on Delivery', 'E wallet']
Length: 5, dtype: str
\n数据类型:
CustomerID                       int64
Churn                            int64
Tenure                         float64
PreferredLoginDevice               str
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode               str
Gender                             str
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferedOrderCat                   str
SatisfactionScore                int64
MaritalStatus                      str
NumberOfAddress 

---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [8]:
# 5. 数据转换与候选异常值检查

def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }


# 1. 创建 TenureGroup（用户使用时长分层）
tenure_bins = [0, 6, 12, 24, 36, float('inf')]
tenure_labels = ['0-6个月', '6-12个月', '12-24个月', '24-36个月', '36个月以上']

cleaned_df['TenureGroup'] = pd.cut(cleaned_df['Tenure'], bins=tenure_bins, labels=tenure_labels, right=False)

print("\\nTenureGroup 分布:")
print(cleaned_df['TenureGroup'].value_counts().sort_index())


# 2. 创建 IsMobileLogin（是否主要使用移动端登录）
# 移动端包括: Mobile Phone, Phone
cleaned_df['IsMobileLogin'] = cleaned_df['PreferredLoginDevice'].apply(
    lambda x: 1 if x in ['Mobile Phone', 'Phone'] else 0
)

print("\\nIsMobileLogin 分布:")
print(cleaned_df['IsMobileLogin'].value_counts())


# 3. 候选异常值报告
outlier_cols = ['WarehouseToHome', 'OrderCount', 'CashbackAmount']
outlier_report = {}

print("\\n候选异常值报告:")
print("="*60)

for col in outlier_cols:
    if col in cleaned_df.columns:
        summary = iqr_outlier_summary(cleaned_df[col])
        outlier_report[col] = summary
        print(f"\\n{col}:")
        print(f"  Q1: {summary['Q1']:.2f}")
        print(f"  Q3: {summary['Q3']:.2f}")
        print(f"  IQR: {summary['Q3'] - summary['Q1']:.2f}")
        print(f"  下限: {summary['下限']:.2f}")
        print(f"  上限: {summary['上限']:.2f}")
        print(f"  候选异常值数量: {summary['候选异常值数量']}")

# 转换为DataFrame便于查看
outlier_df = pd.DataFrame(outlier_report).T
print("\\n异常值报告汇总:")
print(outlier_df)

# 保存异常值报告
outlier_df.to_csv(OUTPUT_DIR / 'outlier_report.csv')
print(f"\\n异常值报告已保存至: {OUTPUT_DIR / 'outlier_report.csv'}")

# 查看添加的新列
print("\\n新增列预览:")
print(cleaned_df[['Tenure', 'TenureGroup', 'PreferredLoginDevice', 'IsMobileLogin']].head())

# 保存最终数据
cleaned_df.to_csv(OUTPUT_DIR / 'e_commerce_cleaned_final.csv', index=False)
print(f"\\n最终清洗数据已保存至: {OUTPUT_DIR / 'e_commerce_cleaned_final.csv'}")

\nTenureGroup 分布:
TenureGroup
0-6个月      1967
6-12个月     1585
12-24个月    1574
24-36个月     500
36个月以上        4
Name: count, dtype: int64
\nIsMobileLogin 分布:
IsMobileLogin
1    3996
0    1634
Name: count, dtype: int64
\n候选异常值报告:
\nWarehouseToHome:
  Q1: 9.00
  Q3: 20.00
  IQR: 11.00
  下限: -7.50
  上限: 36.50
  候选异常值数量: 2
\nOrderCount:
  Q1: 1.00
  Q3: 3.00
  IQR: 2.00
  下限: -2.00
  上限: 6.00
  候选异常值数量: 703
\nCashbackAmount:
  Q1: 145.77
  Q3: 196.39
  IQR: 50.62
  下限: 69.84
  上限: 272.33
  候选异常值数量: 438
\n异常值报告汇总:
                    Q1     Q3    下限     上限  候选异常值数量
WarehouseToHome   9.00  20.00 -7.50  36.50     2.00
OrderCount        1.00   3.00 -2.00   6.00   703.00
CashbackAmount  145.77 196.39 69.84 272.33   438.00
\n异常值报告已保存至: D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24040087\output\day04_project\outlier_report.csv
\n新增列预览:
   Tenure TenureGroup PreferredLoginDevice  IsMobileLogin
0    4.00       0-6个月         Mobile Phone              1
1    9.00      6-12个月         M

### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [9]:
# 任务 4：业务规则检查

print("\\n" + "="*60)
print("业务规则检查")
print("="*60)

# 检查不合规记录
rules = {
    '使用时长为负 (Tenure < 0)': (cleaned_df['Tenure'] < 0).sum(),
    '仓库距离为负 (WarehouseToHome < 0)': (cleaned_df['WarehouseToHome'] < 0).sum(),
    '订单数 <= 0 (OrderCount <= 0)': (cleaned_df['OrderCount'] <= 0).sum(),
    '返现金额为负 (CashbackAmount < 0)': (cleaned_df['CashbackAmount'] < 0).sum()
}

business_rule_report = pd.DataFrame({
    "规则": list(rules.keys()),
    "不合规记录数": list(rules.values())
})

print("\\n业务规则检查报告:")
display(business_rule_report)

# 详细查看不符合规则的记录（如果存在）
for rule, count in rules.items():
    if count > 0:
        col = rule.split('(')[1].split(')')[0]
        print(f"\\n⚠ 发现 {count} 条 {rule}")
        print(cleaned_df[cleaned_df[col] < 0][[col, 'CustomerID', 'Churn']].head())

# 处理结论
print("\\n" + "="*60)
print("处理结论:")
print("="*60)

if business_rule_report['不合规记录数'].sum() == 0:
    print("✓ 所有业务规则检查通过，未发现不合规记录。数据质量良好。")
else:
    print("⚠ 发现不合规记录，处理方案如下：")
    for rule, count in rules.items():
        if count > 0:
            print(f"  - {rule}: 共 {count} 条")
    print("\\n建议处理方案：")
    print("  1. 此类记录数量较少，建议保留但进行标记，供后续分析参考")
    print("  2. 或者根据业务理解将其修正为合理的边界值（如：Tenure < 0 改为 0）")
    print("  3. 本项目仅做记录，不自动删除或修正")

# 保存业务规则检查报告
business_rule_report.to_csv(OUTPUT_DIR / 'business_rule_report.csv', index=False)
print(f"\\n业务规则报告已保存至: {OUTPUT_DIR / 'business_rule_report.csv'}")

\n============================================================
业务规则检查
\n业务规则检查报告:


,规则,不合规记录数
0,使用时长为负 (Tenure < 0),0
1,仓库距离为负 (WarehouseToHome < 0),0
2,订单数 <= 0 (OrderCount <= 0),0
3,返现金额为负 (CashbackAmount < 0),0


\n============================================================
处理结论:
✓ 所有业务规则检查通过，未发现不合规记录。数据质量良好。
\n业务规则报告已保存至: D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24040087\output\day04_project\business_rule_report.csv


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [10]:
# 6. 项目验收与交付

print("\\n" + "="*60)
print("项目验收与交付")
print("="*60)

# 生成清洗后质量报告
quality_after = build_quality_report(cleaned_df)

# 验收检查
print("\\n验收检查:")
print("-"*40)

# 1. 检查数值列是否还有缺失值（只检查存在的列）
existing_numeric_cols = [col for col in NUMERIC_MISSING_COLS if col in cleaned_df.columns]
if existing_numeric_cols:
    missing_check = cleaned_df[existing_numeric_cols].isna().sum().sum()
    if missing_check == 0:
        print("✓ 数值列无缺失值")
    else:
        print(f"✗ 数值列仍有 {missing_check} 个缺失值")
else:
    print("⚠ 未找到需要检查的数值列")

# 2. 检查类别是否已标准化
if "PreferredLoginDevice" in cleaned_df.columns:
    if "Phone" not in cleaned_df["PreferredLoginDevice"].unique():
        print("✓ 'PreferredLoginDevice' 已标准化")
    else:
        print("✗ 'PreferredLoginDevice' 未完全标准化")

if "PreferredPaymentMode" in cleaned_df.columns:
    if "COD" not in cleaned_df["PreferredPaymentMode"].unique() and "CC" not in cleaned_df["PreferredPaymentMode"].unique():
        print("✓ 'PreferredPaymentMode' 已标准化")
    else:
        print("✗ 'PreferredPaymentMode' 未完全标准化")

# 3. 检查新增列是否存在
if {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns):
    print("✓ 新增列 'TenureGroup' 和 'IsMobileLogin' 已创建")
else:
    missing_cols = {"TenureGroup", "IsMobileLogin"} - set(cleaned_df.columns)
    print(f"✗ 缺少新增列: {missing_cols}")

# 4. 检查是否还有重复行
duplicate_check = cleaned_df.duplicated().sum()
if duplicate_check == 0:
    print("✓ 无重复行")
else:
    print(f"✗ 仍有 {duplicate_check} 行重复")

print("\\n" + "="*60)
print("导出交付文件:")
print("="*60)

# 导出所有交付文件
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", encoding="utf-8-sig")
print(f"✓ 清洗前质量报告: {OUTPUT_DIR / 'data_quality_before.csv'}")

quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", encoding="utf-8-sig")
print(f"✓ 清洗后质量报告: {OUTPUT_DIR / 'data_quality_after.csv'}")

cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
print(f"✓ 清洗日志: {OUTPUT_DIR / 'cleaning_log.csv'}")

cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
print(f"✓ 清洗后数据: {OUTPUT_DIR / 'ecommerce_customer_cleaned.csv'}")

# 导出异常值报告
if 'outlier_df' in locals():
    outlier_df.to_csv(OUTPUT_DIR / "outlier_report.csv", encoding="utf-8-sig")
    print(f"✓ 异常值报告: {OUTPUT_DIR / 'outlier_report.csv'}")

# 导出业务规则检查报告
if 'business_rule_report' in locals():
    business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")
    print(f"✓ 业务规则报告: {OUTPUT_DIR / 'business_rule_report.csv'}")

# 清洗前后对比汇总
print("\\n" + "="*60)
print("清洗前后对比汇总:")
print("="*60)

# 只对比都存在的列
common_cols = [col for col in quality_before.index if col in quality_after.index]
comparison = pd.DataFrame({
    '清洗前': quality_before.loc[common_cols, '缺失数量'],
    '清洗后': quality_after.loc[common_cols, '缺失数量']
})
print(comparison)

print(f"\\n数据量变化: {len(raw_df)} → {len(cleaned_df)} 行")

print("\\n" + "="*60)
print("✅ 项目验收通过！所有交付物已生成。")
print(f"交付目录: {OUTPUT_DIR}")
print("="*60)

\n============================================================
项目验收与交付
\n验收检查:
----------------------------------------
✓ 数值列无缺失值
✓ 'PreferredLoginDevice' 已标准化
✓ 'PreferredPaymentMode' 已标准化
✓ 新增列 'TenureGroup' 和 'IsMobileLogin' 已创建
✓ 无重复行
\n============================================================
导出交付文件:
✓ 清洗前质量报告: D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24040087\output\day04_project\data_quality_before.csv
✓ 清洗后质量报告: D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24040087\output\day04_project\data_quality_after.csv
✓ 清洗日志: D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24040087\output\day04_project\cleaning_log.csv
✓ 清洗后数据: D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24040087\output\day04_project\ecommerce_customer_cleaned.csv
✓ 异常值报告: D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24040087\output\day04_project\outlier_report.csv
✓ 业务规则报告: D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24

## 项目复盘

请在提交前用不超过 200 字回答：

11. 本项目发现了哪些数据质量问题？

· 数值字段存在缺失值（Tenure、WarehouseToHome等）
· 类别字段存在同义词混用（Phone/Mobile Phone、COD/Cash on Delivery、CC/Credit Card）
· 潜在异常值（仓库距离、订单数、返现金额）

2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？

· 缺失值：使用总体中位数填补，不按Churn分组
· 类别不一致：通过映射字典统一为标准化名称
· 候选异常值：IQR方法识别并记录，不删除，供后续分析

3. 为什么清洗后的数据可以作为第五天分析的输入？

· 数据完整（无缺失值、无重复行）
· 类别统一，便于建模
· 新增TenureGroup和IsMobileLogin特征
· 保留异常值供探索，未做武断删除
4. 哪些处理规则仍需要业务人员确认？

· 候选异常值的处理阈值和删除标准
· TenureGroup的分层边界是否合理
· 类别映射规则是否完整
· 负值记录是否需要修正为0或其他值

## GitHub提交检查

- [ ] Notebook已重启内核并从头运行成功；
- [ ] `output/day04_project/`包含清洗后数据、质量报告、清洗日志和异常/业务规则报告；
- [ ] 原始Excel没有被覆盖；
- [ ] 清洗函数、处理日志和项目复盘均已完成；
- [ ] 已提交并推送到个人GitHub仓库。